# Python Internal Behaviour

A reference notebook for understanding Python object behavior and several implementation details that can appear surprising.

## Topics

- Mutable vs immutable objects
- Object identity vs equality: `is` vs `==`
- Aliasing
- Shallow copy vs deep copy
- Reference counting
- Garbage collection
- Python integer caching
- String interning
- Function argument passing and object references
- `None` and singleton objects

> **Implementation note:** Python's language semantics are broader than CPython's implementation details. Reference counting, integer caching, and string interning behavior discussed here should not be treated as universal guarantees across every Python implementation.

## 1. Objects, Types, Values, and Identity

In Python, variables are names bound to objects.

An object has:

- a **type**
- a **value/state**
- an **identity**

`type(x)` reports the object's type.

`id(x)` returns an integer identifying the object during its lifetime. In CPython, this is typically related to the object's memory address, but code should not depend on that representation.

In [1]:
x = [1, 2, 3]

print(type(x))
print(id(x))
print(x)

<class 'list'>
2465930287232
[1, 2, 3]


## 2. Mutable vs Immutable Objects

A mutable object can be changed after creation.

Common mutable types:

- `list`
- `dict`
- `set`
- most user-defined objects

Common immutable types:

- `int`
- `float`
- `complex`
- `bool`
- `str`
- `tuple`
- `frozenset`
- `bytes`

Immutability means the object's value/state cannot be changed in place. A variable can still be rebound to a different object.

In [2]:
# Mutable
numbers = [1, 2, 3]
numbers.append(4)
print(numbers)

# Immutable
value = 10
value += 1
print(value)

[1, 2, 3, 4]
11


## 3. Mutability Does Not Mean a Variable Cannot Change

For immutable objects, operations that appear to modify the value usually create a new object and rebind the variable.

For mutable objects, methods can often modify the existing object in place.

In [3]:
text = "hello"
old_id = id(text)

text += " world"

print(text)
print("ID changed:", id(text) != old_id)

items = [1, 2]
old_id = id(items)

items.append(3)

print(items)
print("Same list object:", id(items) == old_id)

hello world
ID changed: True
[1, 2, 3]
Same list object: True


## 4. `==` vs `is`

`==` tests **value equality** by invoking equality comparison.

`is` tests **object identity**: whether two names refer to the exact same object.

Use `==` when comparing values.

Use `is` when identity itself matters, especially when checking against singleton objects such as `None`.

In [4]:
a = [1, 2, 3]
b = [1, 2, 3]
c = a

print(a == b)  # Same value
print(a is b)  # Different objects

print(a == c)
print(a is c)  # Same object

True
False
True
True


## 5. Why `is` Should Not Replace `==`

Two objects can be equal without being identical.

Some immutable objects may be reused or interned by the implementation, which can make `is` appear to work for value comparisons in some cases.

That behavior is not a reason to use `is` for ordinary value comparison.

In [5]:
a = [1000]
b = [1000]

print(a == b)
print(a is b)

# Correct principle:
# use == for values
# use is for identity

True
False


## 6. Aliasing

Aliasing occurs when multiple names refer to the same mutable object.

Changing the object through one name is visible through the other name.

In [6]:
original = [1, 2, 3]
alias = original

alias.append(4)

print(original)
print(alias)
print(original is alias)

[1, 2, 3, 4]
[1, 2, 3, 4]
True


## 7. Aliasing with Dictionaries

The same behavior applies to mutable dictionaries.

In [7]:
config = {"learning_rate": 0.01}
other_name = config

other_name["batch_size"] = 32

print(config)
print(other_name)
print(config is other_name)

{'learning_rate': 0.01, 'batch_size': 32}
{'learning_rate': 0.01, 'batch_size': 32}
True


## 8. Assignment Does Not Copy an Object

An assignment such as:

```python
b = a
```

binds another name to the same object. It does not create a copy.

In [8]:
a = [1, 2, 3]
b = a

print(id(a))
print(id(b))
print(a is b)

2465930441088
2465930441088
True


## 9. Shallow Copy

A shallow copy creates a new outer container, but nested objects are still shared.

Common shallow-copy mechanisms include:

- `list.copy()`
- `dict.copy()`
- slicing
- `copy.copy()`

In [9]:
import copy

original = [[1, 2], [3, 4]]
shallow = copy.copy(original)

print(original is shallow)
print(original[0] is shallow[0])

shallow[0].append(99)

print(original)
print(shallow)

False
True
[[1, 2, 99], [3, 4]]
[[1, 2, 99], [3, 4]]


## 10. Shallow Copy of a Flat List

For a list containing immutable values, a shallow copy often behaves exactly as expected because there are no nested mutable objects to share.

In [10]:
numbers = [1, 2, 3]

copy_numbers = numbers.copy()

copy_numbers.append(4)

print(numbers)
print(copy_numbers)
print(numbers is copy_numbers)

[1, 2, 3]
[1, 2, 3, 4]
False


## 11. Deep Copy

A deep copy recursively copies nested objects where appropriate.

Use `copy.deepcopy()` when independent nested mutable structures are required.

In [11]:
import copy

original = [[1, 2], [3, 4]]
deep = copy.deepcopy(original)

print(original is deep)
print(original[0] is deep[0])

deep[0].append(99)

print(original)
print(deep)

False
False
[[1, 2], [3, 4]]
[[1, 2, 99], [3, 4]]


## 12. Shallow vs Deep Copy

For nested mutable data:

```text
assignment -> same outer object
shallow copy -> new outer object, shared nested objects
deep copy -> recursively independent nested objects
```

In [12]:
import copy

data = [["A"], ["B"]]

alias = data
shallow = copy.copy(data)
deep = copy.deepcopy(data)

print(data is alias)
print(data is shallow)
print(data[0] is shallow[0])
print(data[0] is deep[0])

True
False
True
False


## 13. Copying Dictionaries

`dict.copy()` creates a shallow copy.

Nested mutable values remain shared.

In [13]:
original = {
    "model": "linear",
    "params": {"alpha": 0.1}
}

copied = original.copy()

copied["params"]["alpha"] = 0.5

print(original)
print(copied)
print(original["params"] is copied["params"])

{'model': 'linear', 'params': {'alpha': 0.5}}
{'model': 'linear', 'params': {'alpha': 0.5}}
True


## 14. Deep Copy of a Dictionary

In [14]:
import copy

original = {
    "model": "linear",
    "params": {"alpha": 0.1}
}

copied = copy.deepcopy(original)

copied["params"]["alpha"] = 0.5

print(original)
print(copied)
print(original["params"] is copied["params"])

{'model': 'linear', 'params': {'alpha': 0.1}}
{'model': 'linear', 'params': {'alpha': 0.5}}
False


## 15. Function Argument Passing

Python uses **call by sharing** (also described as passing object references).

A function receives references to the same objects supplied by the caller.

There is no separate pass-by-reference variable mechanism like in some languages, and there is no automatic copy of arguments.

In [15]:
def add_item(items):
    items.append("new")

values = ["old"]

add_item(values)

print(values)

['old', 'new']


## 16. Rebinding a Function Parameter

Reassigning a parameter only changes what that local parameter name refers to. It does not rebind the caller's variable.

In [16]:
def rebind(items):
    items = ["completely", "new"]

values = ["original"]

rebind(values)

print(values)

['original']


## 17. Mutation vs Rebinding in Functions

This distinction is one of the most important parts of Python argument behavior.

- Mutating the object affects other references to that object.
- Rebinding the parameter does not affect the caller's variable.

In [17]:
def demonstrate(items):
    items.append("mutated")
    items = ["rebound"]

values = ["start"]

demonstrate(values)

print(values)

['start', 'mutated']


## 18. Immutable Function Arguments

When an immutable object is passed, operations that appear to change it create/rebind a new object instead of mutating the original.

In [18]:
def increment(value):
    value += 1
    print("Inside:", value)

number = 10

increment(number)

print("Outside:", number)

Inside: 11
Outside: 10


## 19. Mutable Default Arguments

Default argument expressions are evaluated once, when the function is defined, not each time the function is called.

Using a mutable object as a default can therefore create shared state between calls.

In [19]:
def add_item(item, items=[]):
    items.append(item)
    return items

print(add_item("A"))
print(add_item("B"))

['A']
['A', 'B']


## 20. Safer Mutable Defaults

Use `None` as the default and create a new mutable object inside the function.

In [20]:
def add_item(item, items=None):
    if items is None:
        items = []

    items.append(item)
    return items

print(add_item("A"))
print(add_item("B"))

['A']
['B']


# Reference Counting

## 21. Reference Counting in CPython

CPython primarily manages object lifetime using reference counting.

An object's reference count increases when another reference to it is created and decreases when a reference is removed.

When the reference count reaches zero, CPython can usually deallocate the object immediately.

This is a **CPython implementation detail**, not the complete language-level definition of Python object lifetime.

In [21]:
import sys

value = []

print("Initial:", sys.getrefcount(value))

alias = value
print("After alias:", sys.getrefcount(value))

del alias
print("After del:", sys.getrefcount(value))

Initial: 2
After alias: 3
After del: 2


## 22. Why `sys.getrefcount()` Can Be Surprising

`sys.getrefcount(obj)` temporarily creates another reference while evaluating the function call, so the reported count is typically one higher than the number of references you may expect from your source code.

Reference counts can also vary because of implementation details and temporary references.

In [22]:
import sys

obj = object()

print(sys.getrefcount(obj))

2


## 23. Reference Counts and Function Calls

Passing an object to a function temporarily creates another reference to that object.

The exact observed count should not be treated as stable program behavior.

In [23]:
import sys

obj = []

def inspect_reference(value):
    print(sys.getrefcount(value))

inspect_reference(obj)

3


# Garbage Collection

## 24. Reference Counting Is Not Enough

Reference counting alone cannot reclaim cyclic structures.

For example, two objects can reference each other while no outside name references either object.

CPython's cyclic garbage collector detects and handles such cycles.

In [24]:
import gc

print("Garbage collector enabled:", gc.isenabled())
print("GC thresholds:", gc.get_threshold())

Garbage collector enabled: True
GC thresholds: (2000, 10, 10)


## 25. Creating a Reference Cycle

A self-referential list is an example of a reference cycle.

The list contains a reference to itself.

In [25]:
import gc

items = []
items.append(items)

print(items is items[0])

del items

# The cyclic garbage collector can eventually reclaim the cycle.
print("GC count:", gc.get_count())

True
GC count: (525, 11, 5)


## 26. Manual Garbage Collection

The `gc` module provides controls and inspection tools for Python's cyclic garbage collector.

`gc.collect()` requests a collection.

In [26]:
import gc

collected = gc.collect()

print("Objects collected:", collected)

Objects collected: 7


## 27. Garbage Collection Is Not a General `del` Mechanism

`del name` removes a name binding. It does not directly mean "destroy this object."

An object becomes eligible for reclamation when it is no longer reachable through the relevant references.

The exact timing depends on the Python implementation.

In [27]:
items = [1, 2, 3]

alias = items

del items

print(alias)

del alias

[1, 2, 3]


# Integer Caching

## 28. Integer Object Reuse in CPython

CPython maintains a cache of commonly used small integer objects.

This can make identity comparisons between some integer literals appear to work:

```python
a is b
```

However, integer identity should **never** be used to compare numeric values. Use `==`.

In [28]:
a = 10
b = 10

print(a == b)
print(a is b)

large_a = 1000000
large_b = 1000000

print(large_a == large_b)
print(large_a is large_b)

True
True
True
False


## 29. Why Integer Identity Is Unreliable

Integer caching is an implementation optimization.

The exact set of cached integers and the circumstances in which objects are reused should not be relied upon by application code.

Always write:

```python
a == b
```

when you mean "these integers have the same value."

In [29]:
x = 256
y = 256

print(x == y)

# Correct comparison:
if x == y:
    print("Same numeric value")

True
Same numeric value


# String Interning

## 30. What Is String Interning?

String interning allows identical strings to share an object in some circumstances.

CPython interns certain strings automatically, especially some identifier-like strings, and may reuse other string objects as an optimization.

Interning can reduce memory use and speed certain comparisons.

In [30]:
a = "python"
b = "python"

print(a == b)
print(a is b)

True
True


## 31. Do Not Depend on String Identity

Two equal strings are not required to be the same object.

Constructing a string dynamically can produce a distinct object even when its value is equal to an existing string.

In [31]:
a = "python"
b = "".join(["py", "thon"])

print(a == b)
print(a is b)

True
False


## 32. Explicit String Interning

The `sys.intern()` function can explicitly intern a string.

This is an optimization and should be used only when there is a clear reason.

In [32]:
import sys

a = sys.intern("a_unique_string")
b = sys.intern("a_" + "unique_string")

print(a == b)
print(a is b)

True
True


## 33. Interning and Identity

Interning demonstrates why `is` can produce surprising results with immutable values.

The important rule remains:

- `==` → compare values
- `is` → compare identity

In [33]:
x = sys.intern("feature")
y = sys.intern("feature")

print(x == y)
print(x is y)

True
True


# `None` and Singleton Objects

## 34. `None`

`None` represents the absence of a value or "no result" in many Python APIs.

`None` is a singleton: there is one `None` object in a Python process.

Use:

```python
value is None
```

rather than:

```python
value == None
```

In [34]:
value = None

print(value is None)
print(value == None)
print(id(value))

True
True
140736225866224


## 35. Why `is None` Is Preferred

`None` is a singleton, so identity is exactly what we want to test.

Equality can be overloaded by user-defined objects, making `x == None` potentially behave unexpectedly.

In [35]:
class Strange:
    def __eq__(self, other):
        return True

value = Strange()

print(value == None)
print(value is None)

True
False


## 36. Other Singleton-Like Objects

Python has several special singleton or singleton-like objects.

Common examples include:

- `None`
- `True`
- `False`
- `NotImplemented`
- `Ellipsis` (`...`)

Identity checks are appropriate when an API specifically documents a singleton sentinel.

In [36]:
print(None is None)
print(True is True)
print(False is False)
print(NotImplemented is NotImplemented)
print(Ellipsis is Ellipsis)

True
True
True
True
True


## 37. `NotImplemented`

`NotImplemented` is a special singleton used by binary special methods to indicate that an operation is not implemented for the given operand combination.

It is not the same as `NotImplementedError`.

In [37]:
class Example:
    def __eq__(self, other):
        return NotImplemented

print(Example().__eq__(object()))
print(NotImplemented is NotImplemented)

NotImplemented
True


## 38. `NotImplementedError` Is Different

`NotImplementedError` is an exception class commonly used by base classes or abstract-style methods to indicate that a subclass is expected to implement an operation.

`NotImplemented` is a singleton value used by special methods.

In [38]:
print(type(NotImplemented))

try:
    raise NotImplementedError("Subclass must implement this method.")
except NotImplementedError as error:
    print(error)

<class 'NotImplementedType'>
Subclass must implement this method.


## 39. `Ellipsis`

`Ellipsis` is the singleton object represented by `...`.

It is used by some Python APIs, notably NumPy indexing, and can also be used as a custom sentinel.

In [39]:
print(Ellipsis)
print(... is Ellipsis)

Ellipsis
True


## 40. Custom Sentinel Objects

A unique object can be used as a sentinel when `None` is itself a valid input value.

The sentinel is commonly compared with `is`.

In [40]:
_MISSING = object()

def get_value(value=_MISSING):
    if value is _MISSING:
        return "argument was not supplied"
    return value

print(get_value())
print(get_value(None))
print(get_value(10))

argument was not supplied
None
10


## 41. Object Identity Can Change Across Lifetimes

An `id()` value is only guaranteed to uniquely identify an object while that object is alive.

After an object is destroyed, its identity value may potentially be reused for a later object.

In [41]:
a = object()
first_id = id(a)

del a

b = object()

print("First ID:", first_id)
print("Second ID:", id(b))

First ID: 2465928861504
Second ID: 2465928861568


## 42. Immutable Container vs Mutable Contents

An immutable container can still contain mutable objects.

A tuple cannot have its element reference replaced, but an embedded list can still be mutated.

In [42]:
data = ("model", {"learning_rate": 0.01})

data[1]["learning_rate"] = 0.1

print(data)

# data[0] = "other"  # TypeError

('model', {'learning_rate': 0.1})


## 43. Hashability and Immutability

Hashability and immutability are related but not identical concepts.

A hashable object has a hash value that remains stable during its lifetime and can generally be used as a dictionary key or set element.

A tuple is hashable only when its contents are hashable.

In [43]:
print(hash((1, "python")))

# This fails because the tuple contains a list:
# hash((1, [2, 3]))

2828491712975162629


## 44. Practical Example: Configuration Aliasing

Aliasing can unintentionally make two configuration objects share nested state.

In [44]:
base_config = {
    "model": {
        "name": "linear",
        "params": {"alpha": 0.1}
    }
}

experiment = base_config.copy()

experiment["model"]["params"]["alpha"] = 1.0

print("Base:", base_config)
print("Experiment:", experiment)

Base: {'model': {'name': 'linear', 'params': {'alpha': 1.0}}}
Experiment: {'model': {'name': 'linear', 'params': {'alpha': 1.0}}}


## 45. Fixing Nested Configuration Aliasing

Use `deepcopy()` when the nested configuration must become independent.

In [45]:
from copy import deepcopy

base_config = {
    "model": {
        "name": "linear",
        "params": {"alpha": 0.1}
    }
}

experiment = deepcopy(base_config)

experiment["model"]["params"]["alpha"] = 1.0

print("Base:", base_config)
print("Experiment:", experiment)

Base: {'model': {'name': 'linear', 'params': {'alpha': 0.1}}}
Experiment: {'model': {'name': 'linear', 'params': {'alpha': 1.0}}}


## 46. Practical Example: Function Mutation

Functions that mutate input collections should make that behavior clear in their API or documentation.

In [46]:
def add_feature(features, name):
    features.append(name)

features = ["age", "income"]

add_feature(features, "score")

print(features)

['age', 'income', 'score']


## 47. Practical Example: Returning a New Object

A function can instead return a new object, leaving the original unchanged.

In [47]:
def with_feature(features, name):
    return features + [name]

features = ["age", "income"]
updated = with_feature(features, "score")

print("Original:", features)
print("Updated:", updated)

Original: ['age', 'income']
Updated: ['age', 'income', 'score']


## 48. Common Misconceptions

### Misconception 1
`b = a` copies an object.

**Reality:** it creates another reference to the same object.

### Misconception 2
`is` means "same value".

**Reality:** `is` means "same object".

### Misconception 3
A tuple is always deeply immutable.

**Reality:** a tuple can contain mutable objects.

### Misconception 4
`del x` always destroys an object.

**Reality:** it removes a name/reference. The object may still have other references.

### Misconception 5
Integer caching and string interning make `is` safe for values.

**Reality:** these are implementation optimizations; use `==` for value comparison.

## 49. Language Guarantees vs CPython Details

### Python-level concepts
- Mutable vs immutable behavior
- Equality vs identity
- Aliasing
- Function argument semantics
- `None` identity
- Copy semantics

### CPython implementation details
- Reference counting as the primary immediate lifetime mechanism
- Specific integer caches
- Particular string interning behavior
- Some details of object allocation and deallocation

Other Python implementations can make different implementation choices while preserving Python's language semantics.

## 50. Quick Reference

| Concept | Key idea |
|---|---|
| Mutable | Object can change in place |
| Immutable | Object value cannot change in place |
| `==` | Value/equality comparison |
| `is` | Object identity comparison |
| Aliasing | Multiple names reference one object |
| Assignment | Binds another name; does not copy |
| Shallow copy | New outer object, nested references may be shared |
| Deep copy | Recursively copies nested structures |
| Function arguments | Call by sharing/object-reference semantics |
| Reference counting | Major CPython lifetime mechanism |
| GC | Handles cyclic garbage in CPython |
| Integer caching | CPython optimization; do not rely on identity |
| String interning | Shared immutable string objects in some cases |
| `None` | Singleton; use `is None` |
| Sentinel | Unique object commonly checked with `is` |

## Common Rules to Remember

```python
# Compare values
a == b

# Compare identity
a is b

# Check for None
value is None

# Create an independent nested copy
copy.deepcopy(value)

# Remember that assignment does not copy
b = a

# Safe mutable default
def func(value=None):
    if value is None:
        value = []
```

The most important distinction is:

> **Names refer to objects. Objects have identity and state. Mutability determines whether that state can change in place.**